# Grid Stability Analysis and Model Training

This notebook now serves as a high-level driver for the data processing and model training pipeline. The core logic has been refactored into modules within the `src` directory.

In [7]:
# Add project root to path to allow importing from src
import sys
from pathlib import Path
project_root = str(Path().resolve().parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(f"Project root added to path: {project_root}")

Project root added to path: C:\Users\fatem\OneDrive - University of East London\LEVEL 6\CN6000\Final Docs\Implementation


In [8]:
# Install all required dependencies
import sys
!{sys.executable} -m pip install --user lightgbm tensorflow polars openmeteo-requests requests-cache retry scikit-learn swingingdoor matplotlib

  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
  Using cached tensorflow-2.20.0-cp313-cp313-win_amd64.whl.metadata (4.6 kB)


ERROR: Could not find a version that satisfies the requirement swingingdoor (from versions: none)

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for swingingdoor


## ⚠️ Action Required: Restart the Kernel

After the installation is complete, you **must** restart the kernel for the changes to take effect.

**In Jupyter Notebook/Lab:** Go to the `Kernel` menu and select `Restart`.

Once the kernel has restarted, you can run the cells below.

In [1]:
# Import the necessary functions from our modules
import polars as pl
from src.config import (
    FREQUENCY_DATA_FILE, INERTIA_DATA_FILE, 
    LGBM_MODEL_PATH, LSTM_MODEL_PATH, SCALER_PATH, DEMO_DATA_PATH,
    LGBM_QUANTILE_LOWER_PATH, LGBM_QUANTILE_UPPER_PATH, QUANTILE_ALPHAS
)
from src.data_loader import load_frequency_data, fetch_weather_data, load_inertia_data
from src.feature_engineering import merge_datasets, create_features
from src.model_trainer import train_and_evaluate_lgbm_classifier, train_quantile_model, train_lstm_model
import joblib
import os

ModuleNotFoundError: No module named 'src'

## 1. Data Loading

In [ ]:
df_freq = load_frequency_data(FREQUENCY_DATA_FILE)
df_weather = fetch_weather_data("2019-08-01", "2019-08-31")
df_inertia = load_inertia_data(INERTIA_DATA_FILE)

## 2. Feature Engineering

In [ ]:
df_merged = merge_datasets(df_freq, df_weather, df_inertia)
df_processed = create_features(df_merged)

## 3. Model Training: LightGBM Classifier

In [ ]:
lgbm_classifier, _, _ = train_and_evaluate_lgbm_classifier(df_processed)

## 4. Model Training: LightGBM Quantile Regressors

In [ ]:
lower_model = train_quantile_model(df_processed, alpha=QUANTILE_ALPHAS[0])
upper_model = train_quantile_model(df_processed, alpha=QUANTILE_ALPHAS[1])

## 5. Model Training: LSTM

In [ ]:
lstm_model, scaler = train_lstm_model(df_processed)

## 6. Export Models and Data for Dashboard

In [ ]:
print("Exporting models and data...")
joblib.dump(lgbm_classifier, LGBM_MODEL_PATH)
joblib.dump(lower_model, LGBM_QUANTILE_LOWER_PATH)
joblib.dump(upper_model, LGBM_QUANTILE_UPPER_PATH)
lstm_model.save(LSTM_MODEL_PATH)
joblib.dump(scaler, SCALER_PATH)
print("All models saved.")